2.1 

 1. 输入：$C_{in}=3,\ H_{in}=W_{in}=32$，卷积核$k_h=k_w=5$，$P=2,\ S=2$，输出通道=卷积核个数$=16$。
输出尺寸公式：
$$
H_{out}=\left\lfloor \frac{H_{in}+2P-k_h}{S}\right\rfloor+1,\quad
W_{out}=\left\lfloor \frac{W_{in}+2P-k_w}{S}\right\rfloor+1
$$

$$
H_{out}=\left\lfloor \frac{32+2\times2-5}{2}\right\rfloor+1=\left\lfloor 15.5\right\rfloor+1=16
$$
$$
W_{out}=16
$$
输出尺寸：$\boldsymbol{16\times16\times16}$。

2.单个输出像素乘法次数：
$$
Num = C_{in}\times k_h\times k_w=3\times5\times5=75
$$

In [5]:
# 2.2
import numpy as np

def max_pool2d(input, kernel_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    input: (N, C, H, W) 或 (C, H, W) 的 numpy 数组
    kernel_size: int 或 (kh, kw)
    stride: int 或 (sh, sw)，默认等于 kernel_size
    padding: int 或 (ph, pw)
    返回池化后的 numpy 数组
    """
    # 统一形状为 (N, C, H, W)
    if input.ndim == 3:
        input = input[np.newaxis, ...]
    N, C, H, W = input.shape
    
    # 处理参数
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if stride is None:
        sh = sw = kh
    elif isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    # 填充
    pad_input = np.pad(input, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=float('-inf'))
    
    # 输出尺寸
    H_out = (H + 2*ph - kh) // sh + 1
    W_out = (W + 2*pw - kw) // sw + 1
    
    output = np.zeros((N, C, H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            window = pad_input[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2,3))
    return output.squeeze() if input.shape[0] == 1 else output

# 示例
if __name__ == "__main__":
    x = np.random.randn(1, 3, 32, 32)
    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("手动池化输出形状:", out.shape)

手动池化输出形状: (3, 16, 16)


3.1 

输入和输出特征图通道数均为 $C$。

1. 一个 $5 \times 5$ 卷积层（不带偏置）的参数量
   参数量 $= C \times C \times 5 \times 5 = \mathbf{25C^2}$。

2. 两个串联的 $3 \times 3$ 卷积层（不带偏置，每层通道数 $C$）的总参数量  
   每个 $3\times3$ 卷积层参数量 $= C \times C \times 3 \times 3 = 9C^2$。  
   两层总计 $\mathbf{18C^2}$。

In [6]:
# 3.2
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.block(x)

# 示例：输入通道3，输出16，卷积核3x3，步幅1，填充1
block = NiNBlock(3, 16, 3, 1, 1)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print("NiN块输出形状:", out.shape)

NiN块输出形状: torch.Size([1, 16, 32, 32])


4.1 

$x_1=2,x_2=4,x_3=6,x_4=8,\gamma=2,\beta=1,\epsilon=0$
$$
\mu=\frac{x_1+x_2+x_3+x_4}{4}=\frac{2+4+6+8}{4}=5
$$
$$
\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}=5,\quad \sigma=\sqrt{5}
$$
$$
y_i=\gamma\cdot\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta=2\cdot\frac{x_i-5}{\sqrt{5}}+1
$$

$$
y_1=1-\frac{6}{\sqrt{5}},\quad
y_2=1-\frac{2}{\sqrt{5}},\quad
y_3=1+\frac{2}{\sqrt{5}},\quad
y_4=1+\frac{6}{\sqrt{5}}
$$

近似值：$y_1\approx-1.683,\ y_2\approx0.106,\ y_3\approx1.894,\ y_4\approx3.683$。


In [7]:
# 4.2
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        # 主路径：两个 3x3 卷积，每个后跟 BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 捷径连接
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = None
    
    def forward(self, x):
        identity = x
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.shortcut is not None:
            identity = self.shortcut(x)
        y += identity
        return self.relu(y)

# 示例
res_block = Residual(3, 16, use_1x1conv=True)
x = torch.randn(1, 3, 32, 32)
out = res_block(x)
print("残差块输出形状:", out.shape)

残差块输出形状: torch.Size([1, 16, 32, 32])


5.1 

1. 
   底层特征（边缘、纹理等）具有通用性，在源数据集（如 ImageNet）上学到的知识对大多数视觉任务有益，无需大幅改变；顶层输出层与具体任务相关，随机初始化后需要快速适应新数据集，因此采用较大学习率。

2. 
   (1)冻结所有特征提取层（预训练部分），仅微调顶层分类器。  
   (2)使用很小的学习率（如 $10^{-4}$ 或更低）。  
   (3)减少训练轮数（early stopping）。  
   (4)使用更强的正则化（如 Dropout、权重衰减）。

In [8]:
# 5.2
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])


6.1 

$A=[10,10,50,50],\ B=[30,30,70,70]$
$$
S_A=(50-10)\times(50-10)=1600,\quad S_B=(70-30)\times(70-30)=1600
$$
交集坐标：$x_{min}=\max(10,30)=30,x_{max}=\min(50,70)=50,y_{min}=30,y_{max}=50$
$$
S_{inter}=(50-30)\times(50-30)=400
$$
$$
S_{union}=S_A+S_B-S_{inter}=1600+1600-400=2800
$$
$$
IoU=\frac{S_{inter}}{S_{union}}=\frac{400}{2800}=\frac17\approx0.1429
$$

In [9]:
# 6.2
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):
    """
    logits: (N, K) 模型输出未经过 softmax
    labels: (N,) 真实类别索引
    epsilon: 平滑因子
    返回标量损失
    """
    K = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 构建平滑后的目标分布
    smooth_targets = torch.full_like(log_probs, epsilon / (K - 1))
    smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    
    loss = - (smooth_targets * log_probs).sum(dim=-1).mean()
    return loss

# 示例
logits = torch.randn(4, 10)
labels = torch.tensor([1, 3, 5, 0])
loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
print("标签平滑损失:", loss.item())

标签平滑损失: 2.183980941772461
